# Eje 7c — QuantLib (plataforma de pricing)

Derivados Financieros · QUANt UCEMA · **Pricing avanzado**

Basado en el material de clase (`intoquantilib` / `QuantLib_extra`): QuantLib es una
librería C++/Python orientada a objetos para pricing y riesgo. Acá armamos **opción +
proceso BSM + engines** (BS, FD, MC, binomial) y cerramos con americanas + wrappers `Codigo`.

| En este eje | Notebook |
|-------------|----------|
| 07a | Monte Carlo |
| 07b | Diferencias finitas |
| **07c** | QuantLib |
| 07d | Comparación nativo vs QL |

**Plan**
1. Qué es QuantLib
2. Inputs y fechas/calendario
3. Payoff + exercise → `VanillaOption`
4. Spot, curvas, vol → proceso BSM
5. Engines europeos (BS, FD, MC, Binomial) + griegas / error
6. Americanas (BAW, BjS, Bin, FD, MC)
7. Atajo: wrappers `opcion_*_ql`
8. Resumen

## 1) Setup

In [1]:
import math
import sys
import timeit
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
for p in [ROOT, *ROOT.parents]:
    if (p / "Codigo" / "pricing").exists():
        sys.path.insert(0, str(p))
        break

print("OK")

OK


In [2]:
import QuantLib as ql
from Codigo.pricing import (
    _QL_AVAILABLE,
    opcion_europea_bs, opcion_europea_bs_ql,
    opcion_europea_bin, opcion_europea_bin_ql,
    opcion_europea_mc, opcion_europea_mc_ql,
    opcion_europea_fd, opcion_europea_fd_ql,
    opcion_americana_bs, opcion_americana_bs_ql,
    opcion_americana_bin, opcion_americana_bin_ql,
)
print("QuantLib", ql.__version__, "| wrappers:", _QL_AVAILABLE)
assert _QL_AVAILABLE

QuantLib 1.41 | wrappers: True


## 2) Qué es QuantLib (idea)

- Librería open source de finanzas cuantitativas (C++ con bindings Python).
- Separación OOP: **fechas / curvas / proceso / instrumento / engine**.
- Cambiar el motor (BS → MC → FD) sin rearmar la opción: `setPricingEngine(...)`.
- En bancos/fondos suele ser la referencia para validar pricers caseros.

En este repo cada modelo nativo tiene un gemelo `*_ql` con la **misma firma**.

## 3) Inputs del ejemplo (estilo clase)

In [3]:
# Caso didáctico tipo AAPL legacy (fechas fijas) — números redondos de curso
precio_activo = 440.0
precio_ejercicio = 450.0
volatilidad = 0.3394
tasa_interes = 0.0013
tasa_dividendos = 0.075

fecha_valuacion = ql.Date(6, 8, 2020)
fecha_expiracion = ql.Date(16, 10, 2020)
ql.Settings.instance().evaluationDate = fecha_valuacion

day_count = ql.Actual365Fixed()
calendario = ql.UnitedStates(ql.UnitedStates.NYSE)
print("valuacion:", fecha_valuacion, "| expiracion:", fecha_expiracion)
print("daycount fraction T =", day_count.yearFraction(fecha_valuacion, fecha_expiracion))

valuacion: August 6th, 2020 | expiracion: October 16th, 2020
daycount fraction T = 0.19452054794520549


## 4) Instrumento: payoff + exercise → `VanillaOption`

QuantLib distingue **europea** vs **americana** solo en el objeto `Exercise`.

In [4]:
tipo_opcion = ql.Option.Call
payoff = ql.PlainVanillaPayoff(tipo_opcion, precio_ejercicio)
ejercicio_europeo = ql.EuropeanExercise(fecha_expiracion)
opcion_europea = ql.VanillaOption(payoff, ejercicio_europeo)
print(opcion_europea)

<QuantLib.QuantLib.VanillaOption; proxy of <Swig Object of type 'ext::shared_ptr< VanillaOption > *' at 0x0000024D0410F130> >


## 5) Proceso Black–Scholes–Merton

Armamos handles: spot, curva de tasa, curva de dividendos, superficie de vol (acá flat).

In [5]:
objeto_spot = ql.QuoteHandle(ql.SimpleQuote(precio_activo))
print("spot:", objeto_spot.value())

objeto_tasa_interes = ql.YieldTermStructureHandle(
    ql.FlatForward(fecha_valuacion, tasa_interes, day_count)
)
objeto_tasa_dividendos = ql.YieldTermStructureHandle(
    ql.FlatForward(fecha_valuacion, tasa_dividendos, day_count)
)
objeto_volatilidad = ql.BlackVolTermStructureHandle(
    ql.BlackConstantVol(fecha_valuacion, calendario, volatilidad, day_count)
)

print("r forward:", objeto_tasa_interes.forwardRate(
    fecha_valuacion, fecha_expiracion, day_count, ql.Continuous).rate())
print("vol:", objeto_volatilidad.blackVol(fecha_expiracion, precio_ejercicio))

proceso_BSM = ql.BlackScholesMertonProcess(
    objeto_spot, objeto_tasa_dividendos, objeto_tasa_interes, objeto_volatilidad
)
print("proceso OK:", type(proceso_BSM).__name__)

spot: 440.0
r forward: 0.001299999999999829
vol: 0.3394
proceso OK: BlackScholesMertonProcess


## 6a) Engine analítico — Black–Scholes + griegas

In [6]:
opcion_europea.setPricingEngine(ql.AnalyticEuropeanEngine(proceso_BSM))
precio_BS = opcion_europea.NPV()
print("NPV BS:", precio_BS)
print("Delta:", opcion_europea.delta())
print("Gamma:", opcion_europea.gamma())
print("Vega:", opcion_europea.vega())
print("Theta:", opcion_europea.theta())
print("Rho:", opcion_europea.rho())
print("dividendRho:", opcion_europea.dividendRho())
print("thetaPerDay:", opcion_europea.thetaPerDay())
print("strikeSensitivity:", opcion_europea.strikeSensitivity())
print("elasticity:", opcion_europea.elasticity())

NPV BS: 19.018046859717114
Delta: 0.4258321048466277
Gamma: 0.0058826519800777724
Vega: 75.18926359141635
Theta: -51.76161250063861
Rho: 32.74716062566774
dividendRho: -36.446561521667505
thetaPerDay: -0.141812636988051
strikeSensitivity: -0.374106842828442
elasticity: 9.852017271520342


## 6b) Engine FD — `FdBlackScholesVanillaEngine`

Se puede elegir esquema (`MethodOfLines`, etc.) y grillas $t$ / $x$.

In [7]:
tGrid, xGrid, dampingSteps = 100, 100, 0
esquema = ql.FdmSchemeDesc.MethodOfLines()
opcion_europea.setPricingEngine(
    ql.FdBlackScholesVanillaEngine(proceso_BSM, tGrid, xGrid, dampingSteps, esquema)
)
precio_DF = opcion_europea.NPV()
print("NPV FD:", precio_DF)
print("Delta/Theta/Gamma FD:", opcion_europea.delta(), opcion_europea.theta(), opcion_europea.gamma())

NPV FD: 19.025864750659146
Delta/Theta/Gamma FD: 0.42612664943613715 -52.022379981384326 0.005887126552554738


## 6c) Engine Monte Carlo — `MCEuropeanEngine` + errorEstimate

In [8]:
opcion_europea.setPricingEngine(
    ql.MCEuropeanEngine(
        proceso_BSM,
        "PseudoRandom",
        timeSteps=20,
        requiredSamples=80_000,
        seed=42,
    )
)
precio_MC = opcion_europea.NPV()
print("NPV MC:", precio_MC)
print("errorEstimate:", opcion_europea.errorEstimate())

NPV MC: 19.130355599413765
errorEstimate: 0.12692929012763146


## 6d) Engine binomial — `BinomialVanillaEngine`

Árboles típicos: `crr`, `jr`, `eqp`, `trigeorgis`, `tian`, `lr` (Leisen–Reimer), …

In [9]:
opcion_europea.setPricingEngine(ql.BinomialVanillaEngine(proceso_BSM, "crr", 800))
precio_Bin = opcion_europea.NPV()
print("NPV Bin CRR:", precio_Bin)
print("Delta/Theta/Gamma Bin:", opcion_europea.delta(), opcion_europea.theta(), opcion_europea.gamma())

opcion_europea.setPricingEngine(ql.BinomialVanillaEngine(proceso_BSM, "lr", 800))
print("NPV Bin LR:", opcion_europea.NPV())

NPV Bin CRR: 19.016817809736228
Delta/Theta/Gamma Bin: 0.42584634116463643 -51.806435134420774 0.005886712981998227
NPV Bin LR: 18.9898435069753


## 6e) Tabla europea (engines QL)

In [10]:
pd.DataFrame([
    {"engine": "AnalyticEuropean (BS)", "NPV": precio_BS},
    {"engine": "FdBlackScholes", "NPV": precio_DF},
    {"engine": "MCEuropean", "NPV": precio_MC},
    {"engine": "Binomial CRR", "NPV": precio_Bin},
]).round(6)

,engine,NPV
0,AnalyticEuropean (BS),19.018047
1,FdBlackScholes,19.025865
2,MCEuropean,19.130356
3,Binomial CRR,19.016818


## 7) Opciones americanas

Mismo payoff + `AmericanExercise`. BS analítico europeo **no aplica** (QuantLib error).
Usamos BAW / Bjerksund–Stensland, binomial, FD y MC americano.

In [11]:
ejercicio_ame = ql.AmericanExercise(fecha_valuacion, fecha_expiracion)
opcion_ame = ql.VanillaOption(payoff, ejercicio_ame)

# BS europeo sobre americana → debe fallar
try:
    opcion_ame.setPricingEngine(ql.AnalyticEuropeanEngine(proceso_BSM))
    print("NPV (no debería):", opcion_ame.NPV())
except Exception as e:
    print("Esperado — AnalyticEuropean no vale para americana:")
    print(" ", type(e).__name__ + ":", str(e)[:120])

opcion_ame.setPricingEngine(ql.BaroneAdesiWhaleyApproximationEngine(proceso_BSM))
precio_BAW = opcion_ame.NPV()
opcion_ame.setPricingEngine(ql.BjerksundStenslandApproximationEngine(proceso_BSM))
precio_BjS = opcion_ame.NPV()
opcion_ame.setPricingEngine(ql.BinomialVanillaEngine(proceso_BSM, "crr", 800))
precio_Bin_A = opcion_ame.NPV()
opcion_ame.setPricingEngine(ql.FdBlackScholesVanillaEngine(proceso_BSM, 100, 100))
precio_FD_A = opcion_ame.NPV()
opcion_ame.setPricingEngine(
    ql.MCAmericanEngine(proceso_BSM, "PseudoRandom", timeSteps=20, requiredSamples=40_000, seed=1)
)
precio_MC_A = opcion_ame.NPV()

pd.DataFrame([
    {"engine": "BAW approx", "NPV": precio_BAW},
    {"engine": "Bjerksund–Stensland", "NPV": precio_BjS},
    {"engine": "Binomial CRR", "NPV": precio_Bin_A},
    {"engine": "FD", "NPV": precio_FD_A},
    {"engine": "MC American", "NPV": precio_MC_A},
]).round(6)

Esperado — AnalyticEuropean no vale para americana:
  RuntimeError: not an European option


,engine,NPV
0,BAW approx,19.405381
1,Bjerksund–Stensland,19.217979
2,Binomial CRR,19.424814
3,FD,19.412009
4,MC American,19.358601


## 8) Wrappers del curso (`Codigo.pricing.*_ql`)

Misma firma que los nativos — útil para no reinventar fechas en cada notebook.

In [12]:
# Parámetros "juguete" alineados al resto del eje 7
tipo, S, K, T = "C", 100.0, 100.0, 1.0
r, sigma, div = 0.05, 0.25, 0.0
rows = []
for name, fn, fn_ql, args in [
    ("BS", opcion_europea_bs, opcion_europea_bs_ql, (tipo, S, K, T, r, sigma, div)),
    ("Bin", opcion_europea_bin, opcion_europea_bin_ql, (tipo, S, K, T, r, sigma, div, 200)),
    ("MC", opcion_europea_mc, opcion_europea_mc_ql, (tipo, S, K, T, r, sigma, div, 20_000)),
    ("FD", opcion_europea_fd, opcion_europea_fd_ql, (tipo, S, K, T, r, sigma, div, 120)),
    ("Ame Bin", opcion_americana_bin, opcion_americana_bin_ql, ("P", S, K, T, r, sigma, div, 200)),
    ("Ame BAW", opcion_americana_bs, opcion_americana_bs_ql, ("P", S, K, T, r, sigma, div)),
]:
    rows.append({"modelo": name, "nativo": fn(*args), "QL wrapper": fn_ql(*args)})
pd.DataFrame(rows).assign(diff=lambda d: d["nativo"] - d["QL wrapper"]).round(6)

,modelo,nativo,QL wrapper,diff
0,BS,12.335999,12.335999,0.000000
1,Bin,12.323646,12.323367,0.000280
2,MC,12.345990,12.134413,0.211577
3,FD,12.324710,12.341511,-0.016801
4,Ame Bin,7.967099,7.969216,-0.002117
5,Ame BAW,8.000112,7.982520,0.017592


## Resumen

- QuantLib = fechas + handles + proceso + opción + **engine**.
- Europeo: BS da griegas completas; FD/Bin dan Δ,Γ,Θ; MC da `errorEstimate`.
- Americana: no uses AnalyticEuropean; BAW/BjS/Bin/FD/MC sí.
- Validación del curso: nativo vs `*_ql` (eje **07d**).

**Ejercicio:** cambiá el árbol a `"jr"` y `"tian"` (N=500) y compará NPV vs CRR.

**Siguiente:** **07d** — comparación de todos los modelos + tiempos.